# Lab 6.10 &mdash; Connect to External APIs (Google + Wolfram)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Wrap the REAL Google Serper search API as a @tool
- Wrap the REAL Wolfram Alpha compute API as a @tool
- Guard every call so a missing key or API error returns a string, not a crash

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Connecting to real tools & APIs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-10"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Real agents reach the world through **tool integrations** (deck slide 16): **Google Search** (via
[serper.dev](https://serper.dev)) for live facts beyond the training cutoff, **Wolfram Alpha** for exact
computation. The pattern is always: get a key &rarr; wrap the service as a `@tool` &rarr; add it to the
tools list. But **real APIs fail** &mdash; rate limits, timeouts, bad queries, missing keys &mdash; so a
production tool must **guard the call and return a string** instead of crashing the whole agent. Your keys
are loaded from `.env`; if one is missing the tool prints guidance and the run still completes.

In [ ]:
import os
print("SERPER_API_KEY set :", bool(os.getenv("SERPER_API_KEY")))
print("WOLFRAM_ALPHA_APPID set :", bool(os.getenv("WOLFRAM_ALPHA_APPID")))
# Note: this WOLFRAM key is a Short-Answers / LLM-API key -- we call Wolfram's LLM API endpoint,
# the one Wolfram recommends for agents (the langchain_community v2/query wrapper needs a different app type).

## Build it
Wrap the two real APIs as guarded `@tool`s (fill the API call + the error-string fallback), then bind both
to a real agent with `create_agent`.

In [ ]:
import os, urllib.parse, urllib.request
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent

@tool
def google_search(query: str) -> str:
    """Search the web for a current fact or figure. Use for anything not in the model's own memory."""
    if not os.getenv("SERPER_API_KEY"):
        return "search unavailable: set SERPER_API_KEY in .env (free at serper.dev)"
    try:
        return ___   # TODO: GoogleSerperAPIWrapper().run(query) -- the REAL search
    except Exception as e:
        return ___   # TODO: return an error STRING (never raise) -- e.g. "search error: " + type(e).__name__

def _wolfram_llm(query):
    """Call Wolfram's LLM API (the endpoint recommended for agents)."""
    appid = os.getenv("WOLFRAM_ALPHA_APPID")
    url = "https://www.wolframalpha.com/api/v1/llm-api?appid=" + urllib.parse.quote(appid) + "&input=" + urllib.parse.quote(query)
    with urllib.request.urlopen(url, timeout=20) as r:
        return r.read().decode("utf-8", "replace")

@tool
def wolfram(query: str) -> str:
    """Compute or look up an exact quantity: math, unit conversions, science facts. Use for precise computation."""
    if not os.getenv("WOLFRAM_ALPHA_APPID"):
        return "compute unavailable: set WOLFRAM_ALPHA_APPID in .env (developer.wolframalpha.com)"
    try:
        return _wolfram_llm(query)[:400]
    except Exception as e:
        return "compute error: " + type(e).__name__

def build_agent():
    return create_agent(llm, ___)   # TODO: bind both real external tools

try:
    # Direct sanity calls -- these hit the REAL APIs (guarded):
    print("google_search ->", google_search.invoke("population of France")[:120])
    print("wolfram       ->", wolfram.invoke("2400 * 2")[:120])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Bind both real tools to an agent and ask a question that needs a live search and an exact compute. The trace shows real external API calls.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        agent = build_agent()
        result = agent.invoke(
            {"messages": [("user", "Search for the height of the Eiffel Tower in metres, then use wolfram to convert it to feet.")]},
            config={"recursion_limit": 8})
        print_trace(result)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The **direct** `google_search.invoke(...)` / `wolfram.invoke(...)` calls above returned **live** data &mdash; real network, real keys.
- Every call is **guarded**: a missing key or an API error returns a string, so the agent survives. Delete a key from `.env` and the tool prints guidance instead of crashing.
- This is the client's Day-3 "connect agents to Google Search + Wolfram Alpha" lab, for real.

## Your turn (open task &mdash; no grader)
Ask a question that needs only search, then one that needs only compute, then one that needs both, and
compare the traces. **What good looks like:** the agent calls `google_search` for live facts and `wolfram`
for exact math, and a temporarily-broken key (rename it in `.env`) yields a graceful "unavailable" string
rather than a stack trace.

---
### Key takeaway
Get a key -> wrap the service as a guarded @tool -> add it to the list. That is how an agent reaches live Google facts and exact Wolfram computation -- the client's Day-3 external-API lab, for real.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>